# 独立干扰与有效变化
约 600 保留状态 × 3 个核验无关扰动 + 1 个有效变化；禁止用于训练或调参。

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'pilot'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
print('Code:', PROJECT, 'Artifacts:', ROOT)


In [ ]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# Activate editable-install .pth files in this already-running notebook kernel.
import importlib, site
site_dirs = site.getsitepackages()
if site.ENABLE_USER_SITE:
    site_dirs.append(site.getusersitepackages())
for site_dir in dict.fromkeys(site_dirs):
    site.addsitedir(site_dir)
importlib.invalidate_caches()
import sa_jev
print('SA-Jev source:', sa_jev.__file__)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config()
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    from collections import deque
    tail = deque(maxlen=40)
    cmd = [sys.executable, '-u', '-m', 'sa_jev.cli', command, '--root', str(ROOT), *map(str,args)]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                tail.append(line)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
    if returncode:
        raise RuntimeError(f'sajev {command} failed (exit {returncode}). Last output: ' + ''.join(tail))


In [ ]:
# Read the same credential file in every notebook. Fill it in once.
from sa_jev.environment import load_environment
ENV_FILE = (DRIVE if 'google.colab' in sys.modules else PROJECT) / '.env'
credential_status = load_environment(ENV_FILE, template=PROJECT / '.env.example')
print('Credential file:', ENV_FILE)
for name in ('OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY', 'HF_TOKEN'):
    print(name + ':', 'loaded' if credential_status[name] else 'empty')
if not credential_status['OPENAI_API_KEY']:
    print('Fill OPENAI_API_KEY in the file above, save it, then rerun this cell before building the index or answering questions.')


In [ ]:
from sa_jev.experiments import robustness_experiment
rows = robustness_experiment(ROOT, config)

同时检查 JS、Top-1/Top-5、充分性翻转、错误提前停止、nDCG 和有效变化后正确补证。新增已选证据会改变剩余候选集合，不强行比较为不变分布。